# C04 — Inferência local, remota ou privada

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Este notebook corresponde à quarta etapa do projeto da disciplina **"Sistemas Cognitivos
com LLMs"**: comparar três formas de rodar um modelo de linguagem — **local** (na própria
máquina, com Ollama ou Hugging Face), **remota** (pagando por token numa API externa) ou
**privada** (mantendo um servidor próprio, por exemplo uma instância com GPU na AWS) — e
justificar qual faz mais sentido para este projeto. A decisão é usar a **API do Gemini
Flash** como opção remota.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🎯 Objetivos desta etapa**

- **Comparar** três formas de inferência — local (Ollama / Hugging Face), remota (API paga
  por token) e privada (servidor próprio com GPU);
- **Justificar** a escolha da API do Gemini Flash como opção remota;
- **Demonstrar** a inferência remota na prática, com uma chamada real à API;
- **Reunir evidências** critério por critério — privacidade, custo, latência,
  disponibilidade, controle, facilidade de integração, limitações de hardware, dependência
  de internet e risco de exposição de dados;
- **Documentar** as dependências, as variáveis de ambiente e os cuidados de segurança para
  não expor chaves nem dados do projeto.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🌐 Convenções deste notebook**

- **Narrativa, variáveis e descrições em português** — mesma convenção de c01, c02 e c03.
- **Opção remota**: `gemini-3.5-flash` via `google-genai`, com os mesmos parâmetros de
  determinismo dos notebooks anteriores (temperature=0, seed=42 e thinking_budget baixo).
- **Opções comparadas**: local com Ollama ou Hugging Face (como em `c02_prompting.ipynb`),
  remota via API e privada num servidor próprio.
- **Contexto de exemplo**: uma aula já processada em `data/processed/*_portugues.txt`,
  gerada por `c01_modelos_llm.ipynb`.

**Pré-requisito**: executar `c01_modelos_llm.ipynb` antes deste notebook, para gerar
`data/processed/*_portugues.txt`, usado como contexto de exemplo nas chamadas de inferência.

</div>

---

## §1 Justificativa: por que usar a API do Gemini Flash

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**Privacidade**

Ao usar a API do Gemini Flash, os textos do projeto saem do computador do aluno e viajam até os servidores do Google para serem processados. Isso é uma desvantagem real: numa solução local, com Ollama por exemplo, nada sairia da máquina. Para este projeto, porém, o material enviado são transcrições de aulas da própria disciplina, um conteúdo que já é compartilhado entre alunos e professor e não contém dados pessoais sensíveis. O risco de privacidade existe, mas é baixo e aceitável para esse tipo de dado; se o projeto lidasse com informação confidencial, essa escolha teria que ser revista.

**Custo**

O Gemini Flash é o modelo mais barato da família Gemini, escolhido justamente por isso para tarefas simples e de geração curta. Pagamos apenas pelos tokens que realmente usamos, e o volume de um projeto de curso é muito baixo. Para comparar: o Gemini Pro custa cerca de US$1,25 por milhão de tokens de entrada e US$10,00 por milhão de tokens de saída, e mesmo assim os notebooks anteriores mostraram que uma aula inteira traduzida com o Pro saiu por centavos. Já um servidor próprio com GPU, como uma instância AWS g4dn.xlarge, custa entre US$384 e US$400 por mês, fixo, mesmo parado. Pagar por token compensa muito mais nesse volume; o custo fixo só faria sentido com uso intenso e constante.

**Latência**

A latência da API inclui o tempo de ida e volta pela internet, algo que a inferência local não tem. Em compensação, o modelo roda em GPUs muito mais potentes que o hardware do aluno, então o tempo total de resposta costuma ser menor do que rodar um modelo grande numa máquina comum, que pode levar minutos ou nem conseguir carregar o modelo. Para este projeto, onde as chamadas são feitas de forma interativa dentro de um notebook e ninguém depende de resposta em milissegundos, a latência da API é perfeitamente adequada. A desvantagem real é que ela varia conforme a rede e a carga dos servidores, algo fora do nosso controle.

**Disponibilidade**

A API do Google fica disponível o tempo todo, mantida por uma equipe profissional, com redundância que um aluno jamais conseguiria montar sozinho. Um servidor próprio exigiria manutenção, atualização e monitoramento; uma solução local só funciona quando aquele computador específico está ligado e configurado. A desvantagem honesta é a dependência de um serviço externo: se o Google tiver uma instabilidade, mudar a API ou descontinuar o modelo, o projeto para até nos adaptarmos — e isso já aconteceu neste projeto, quando o modelo de embeddings text-embedding-004 foi descontinuado e tivemos que migrar para o gemini-embedding-001. É um risco real, mas administrável num projeto de curso.

**Controle**

Aqui a API perde de forma clara, e é importante admitir isso. Não escolhemos a versão exata do modelo, não podemos ajustar seus pesos, não vemos como ele funciona por dentro, e o Google pode alterá-lo sem aviso. Com um modelo local teríamos controle total sobre versão e comportamento. Para este projeto, esse controle fino não é necessário: usamos o modelo como ele vem, para tarefas comuns de geração e resumo, e os parâmetros que a API expõe — temperatura, limite de tokens, instruções de sistema — são suficientes. Abrimos mão de controle em troca de simplicidade e qualidade, e para o escopo do curso essa troca vale a pena.

**Facilidade de integração**

Este é um dos pontos mais fortes da escolha. A integração se resume a instalar a biblioteca google-genai, carregar a chave de API e fazer chamadas simples em Python, tudo dentro do notebook. Os notebooks anteriores do projeto já usam exatamente esse caminho, então não há nada novo para aprender nem configurar. Uma solução local exigiria baixar modelos de vários gigabytes e gerenciar memória; um servidor privado exigiria configurar máquina, rede, segurança e implantação. A API elimina tudo isso. A pequena desvantagem é ficar preso ao formato e às regras dessa biblioteca específica, mas isso pesa pouco frente ao tempo economizado.

**Limitações de hardware**

Rodar modelos grandes localmente exige GPU com bastante memória, algo que o computador de um aluno normalmente não tem. Modelos pequenos até rodam, como vimos nos notebooks anteriores com DistilGPT-2 e FLAN-T5, mas a qualidade fica visivelmente abaixo. Com a API, o hardware pesado é problema do Google: usamos um modelo de ponta a partir de qualquer máquina, até de um notebook simples. Não há desvantagem real neste critério para o nosso caso; a única ressalva é que essa comodidade nos desobriga de aprender a gerenciar infraestrutura própria, uma habilidade que teria valor em outros contextos.

**Dependência de internet**

Esta é uma desvantagem clara e sem disfarce: sem conexão, nada funciona. Uma solução local com Ollama continuaria rodando offline sem problema. Aceitamos essa dependência porque o projeto inteiro já depende de internet de qualquer forma — baixar modelos do Hugging Face, acessar o repositório, entregar o trabalho — e porque o ambiente de estudo tem conexão estável. Se o projeto precisasse funcionar em campo, sem rede garantida, a escolha teria que ser outra. Para um notebook acadêmico executado em casa ou na faculdade, o risco prático é mínimo. Numa infraestrutura de nuvem como o AWS ECS, essa limitação também deixaria de ser um problema prático: o serviço rodaria com conexão à internet estável e gerenciada pela própria nuvem, diferente de uma máquina pessoal, que pode perder sinal de wifi ou rede.

**Risco de exposição de dados**

O risco tem duas frentes. A primeira é o vazamento da chave de API: se ela for parar num repositório público, alguém pode usá-la e gerar custos na nossa conta. Esse risco é controlado com prática simples: chave guardada num arquivo .env fora do controle de versão, nunca escrita no código nem impressa na saída das células. A segunda frente é o dado enviado à API: tudo que mandamos passa pelos servidores do Google e está sujeito aos termos de uso deles. Como o conteúdo do projeto são transcrições de aula sem informação sigilosa, esse risco é baixo. A regra que adotamos é direta: nada confidencial vai para a API, e a chave é trocada imediatamente se houver qualquer suspeita de vazamento.

</div>

In [1]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

MODELO_GEMINI = "gemini-3.5-flash"


---

## §2 Inferência remota na prática

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Para tornar a escolha concreta, e não só teórica, esta seção mostra a API do Gemini Flash
respondendo de verdade a uma pergunta sobre uma aula real do curso, usando o mesmo texto
processado por `c01_modelos_llm.ipynb`, em dois formatos: texto livre e JSON estruturado,
com parsing da resposta — reaproveitando a mesma função `extrair_json()` já usada em
`c02_prompting.ipynb`, que remove blocos de código ```json``` e localiza o primeiro bloco
JSON válido na saída. A função `gerar` segue o mesmo padrão já usado em c01, c02 e c03:
`temperature=0` e `seed=42` para buscar determinismo, e um orçamento pequeno de raciocínio
interno, já que perguntar sobre um trecho de transcript não exige um pensamento elaborado.

</div>

In [2]:
from pathlib import Path
from google.genai import types
from IPython.display import HTML, display
import re
import json

PROCESSED_DIR = Path("data/processed")
NOME_BASE = "Sistemas_Cognitivos_com_LLMs_aula-01-06-2026_20-08"
CONTEXTO = (PROCESSED_DIR / f"{NOME_BASE}_portugues.txt").read_text(encoding="utf-8").strip()


def gerar(system: str, user: str, max_new_tokens: int = 512) -> str:
    """Chama o Gemini Flash para inferência remota — mesmos parâmetros de determinismo
    de c01, c02 e c03: temperature=0, seed=42, thinking_budget baixo."""
    resposta = client.models.generate_content(
        model=MODELO_GEMINI,
        contents=user,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0,
            seed=42,
            candidate_count=1,
            max_output_tokens=max_new_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=128),
        ),
    )
    return (resposta.text or "").strip()


def extrair_json(texto: str):
    """Remove fences ```json``` e localiza o primeiro bloco JSON; devolve (dado, erro).
    Mesma função já usada em c02_prompting.ipynb."""
    limpo = re.sub(r"```(?:json)?", "", texto).strip()
    inicios = [i for i in (limpo.find("["), limpo.find("{")) if i != -1]
    if not inicios:
        return None, "nenhum bloco JSON encontrado na saída"
    inicio = min(inicios)
    fim = max(limpo.rfind("]"), limpo.rfind("}"))
    if fim <= inicio:
        return None, "bloco JSON incompleto"
    try:
        return json.loads(limpo[inicio:fim + 1]), None
    except json.JSONDecodeError as e:
        return None, f"JSON malformado: {e}"


def mostrar_evidencia(itens, conclusao):
    """Caixa de evidência de um critério: pares rótulo/valor medidos pela célula e a
    conclusão — mesma linguagem visual das caixas markdown do projeto."""
    linhas = "".join(
        f'<p style="margin:0 0 4px 0;"><b>{rotulo}:</b> {valor}</p>'
        for rotulo, valor in itens
    )
    display(HTML(
        '<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;'
        'border-radius:4px;color:#111827;">'
        f'{linhas}'
        f'<p style="margin:8px 0 0 0;"><b>Conclusão:</b> {conclusao}</p>'
        '</div>'
    ))


SYSTEM = (
    "You are an assistant that answers strictly based on the following class transcript, "
    "written in Portuguese. If the transcript does not contain the answer, say so.\n"
    f"<transcript>\n{CONTEXTO}\n</transcript>"
)

# Formato normal: texto livre
pergunta = "Quantas aulas tem o curso ao todo?"
resposta = gerar(SYSTEM, f"Question: {pergunta}\n\nAnswer in Portuguese, in at most 2 sentences.")

# Formato estruturado: JSON, com parsing (mesma extrair_json() de c02_prompting.ipynb)
pergunta_json = (
    "Extract from the transcript: the total number of classes, and the main library used "
    "to access language models. Return ONLY a JSON object like "
    '{"numero_aulas": <int>, "biblioteca_principal": "<string>"}, with no extra text, no '
    "code fences."
)
resposta_json_bruta = gerar(SYSTEM, pergunta_json)
dado_extraido, erro_parsing = extrair_json(resposta_json_bruta)

display(HTML(
    '<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;'
    'border-radius:4px;color:#111827;">'
    '<p style="margin:0 0 4px 0;"><b>Formato normal — texto livre</b></p>'
    f'<p style="margin:0 0 10px 0;"><b>Pergunta:</b> {pergunta}<br>'
    f'<b>Resposta:</b> {resposta}</p>'
    '<p style="margin:0 0 4px 0;"><b>Formato estruturado — JSON</b></p>'
    f'<p style="margin:0;"><b>JSON parseado:</b> '
    f'{dado_extraido if erro_parsing is None else "falhou — " + erro_parsing}</p>'
    '</div>'
))

---

## §3 Evidências por critério



### §3.1 Privacidade

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português
para o espanhol e deixar o aluno hispanofalante fazer perguntas — o que viaja para
fora é justamente a matéria-prima do produto: o texto das aulas.

Não dá para medir privacidade com código, mas dá para mostrar o fato concreto que
sustenta o risco. A célula abaixo resolve, da própria máquina do aluno, o endereço do
endpoint da API: os IPs que aparecem são os servidores públicos do Google para onde
toda chamada abre conexão e envia o transcript inteiro. É a prova em nível de rede —
e não uma frase copiada da documentação — de que o texto sai da máquina e passa a
depender da infraestrutura de um terceiro. Para contraste: numa solução local, o
endereço equivalente seria `127.0.0.1` (localhost) e nada sairia. Também mostramos o
tamanho do que é enviado em cada chamada.

**Veredito para o nosso caso de uso:** este é o critério em que a remota perde — mas perde pouco. O dado enviado são transcrições de aula, um conteúdo já compartilhado entre alunos e professor, sem dado pessoal sensível; para esse tipo de material, a perda de privacidade é pequena e aceitável, e é mais do que compensada pelos ganhos nos demais critérios. Se o SaaS um dia lidar com material confidencial, é este critério que forçaria a revisitar a escolha.

</div>

In [3]:
import socket

# A resolução DNS abaixo é feita da própria máquina do aluno: revela os endereços IP
# públicos da infraestrutura do Google para onde toda chamada da API abre conexão. É a
# prova em nível de rede de que o texto sai da máquina — não uma frase da documentação.
enderecos = socket.getaddrinfo("generativelanguage.googleapis.com", 443)
ips = sorted({item[4][0] for item in enderecos})

itens = [
    ("Endpoint da API do Gemini resolve para", ", ".join(ips)),
    ("De quem são esses endereços", "infraestrutura pública do Google — fora da máquina "
     "e da rede do aluno, sob controle exclusivo deles"),
    ("Para contraste: endpoint de uma solução local (Ollama)",
     "127.0.0.1 (localhost) — o texto nunca sairia da máquina"),
    ("Tamanho do contexto enviado em cada chamada", f"{len(CONTEXTO)} caracteres"),
]
conclusao = (
    "cada chamada abre uma conexão da máquina do aluno para esses IPs do Google e envia o "
    "texto inteiro da aula até lá — endereços de infraestrutura externa, fora do nosso "
    "controle, e é isso que torna a exposição real, não hipotética; o risco em si já está "
    "avaliado na justificativa acima."
)
mostrar_evidencia(itens, conclusao)

### §3.2 Custo

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

A régua de custo deste projeto é a de um **SaaS**: a ideia é um serviço que traduz as
transcrições das aulas do português para o espanhol e deixa o aluno hispanofalante
fazer perguntas em espanhol sobre a aula. Cada pergunta de aluno vira uma chamada de
API como a medida abaixo — ou seja, o custo por chamada é o **custo variável por uso**
do produto (a tradução de uma aula inteira, medida em c01, custa centavos e é paga uma
única vez por aula).

O custo por chamada dá para medir de verdade, usando os dados de uso que a própria API
devolve, e comparar com os preços vigentes em 2026 (tabela oficial em
`ai.google.dev/gemini-api/docs/pricing`, consultada em julho de 2026):

- **Gemini 3.5 Flash** (o modelo deste projeto): US$ 1,50 por milhão de tokens de entrada
  e US$ 9,00 por milhão de saída (entrada em cache: US$ 0,15).
- **Gemini 3.1 Pro**: US$ 2,00 de entrada e US$ 12,00 de saída até 200 mil tokens de
  contexto — e o dobro de entrada (US$ 4,00) acima disso.
- **Gemini 3.1 Flash-Lite** (o piso da família): US$ 0,25 de entrada e US$ 1,50 de saída.

Do outro lado da comparação, manter um servidor próprio com GPU ligado o mês inteiro na
AWS EC2 (on-demand, ~730 h/mês) custa, conforme a instância: **g4dn.xlarge** (T4, 16 GB)
US$ 0,526/h ≈ US$ 384/mês; **g6.xlarge** (L4, 24 GB) US$ 0,80/h ≈ US$ 588/mês;
**g5.xlarge** (A10G, 24 GB) US$ 1,01/h ≈ US$ 734/mês — custo fixo, pago mesmo com a
máquina parada. A célula abaixo mede uma chamada real e faz a conta.

**Veredito para o nosso caso de uso:** pagar por token alinha o custo à receita do SaaS — centavos por aluno ativo contra centenas de dólares fixos por mês de um servidor próprio. Neste critério, a inferência remota não é só conveniente: é a que viabiliza o lucro.

</div>

In [4]:
# Preços oficiais 2026 (ai.google.dev/gemini-api/docs/pricing, consultado em jul/2026)
PRECO_ENTRADA_POR_MILHAO = 1.50   # US$, gemini-3.5-flash, entrada
PRECO_SAIDA_POR_MILHAO = 9.00     # US$, gemini-3.5-flash, saída

# Para comparação, na mesma tabela oficial:
PRECO_PRO_ENTRADA = 2.00          # US$, gemini-3.1-pro (até 200k tokens; dobra acima)
PRECO_PRO_SAIDA = 12.00
PRECO_FLASH_LITE_ENTRADA = 0.25   # US$, gemini-3.1-flash-lite (o piso da família)
PRECO_FLASH_LITE_SAIDA = 1.50

# Servidor próprio com GPU na AWS EC2, on-demand, ~730 h/mês (preços de jul/2026)
SERVIDORES_GPU_MENSAL = {
    "g4dn.xlarge (T4 16GB)": 384,
    "g6.xlarge (L4 24GB)": 588,
    "g5.xlarge (A10G 24GB)": 734,
}
CUSTO_SERVIDOR_MENSAL = min(SERVIDORES_GPU_MENSAL.values())  # a opção mais barata


def gerar_com_metadata(system: str, user: str, max_new_tokens: int = 512):
    """Igual a gerar, mas devolve também o usage_metadata da resposta, para calcular o
    custo real da chamada em dólares."""
    resposta = client.models.generate_content(
        model=MODELO_GEMINI,
        contents=user,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0,
            seed=42,
            candidate_count=1,
            max_output_tokens=max_new_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=128),
        ),
    )
    return (resposta.text or "").strip(), resposta.usage_metadata


pergunta_custo = "Question: Quantos modelos locais este projeto testou em c01?\n\nAnswer in Portuguese, in one sentence."
texto, uso = gerar_com_metadata(SYSTEM, pergunta_custo)
tokens_thinking = uso.thoughts_token_count or 0
custo_chamada = (
    uso.prompt_token_count / 1_000_000 * PRECO_ENTRADA_POR_MILHAO
    + (uso.candidates_token_count + tokens_thinking) / 1_000_000 * PRECO_SAIDA_POR_MILHAO
)
chamadas_por_mes_do_servidor = CUSTO_SERVIDOR_MENSAL / custo_chamada

PERGUNTAS_POR_ALUNO_MES = 100  # hipótese de uso de um aluno ativo do SaaS
custo_aluno_mes = custo_chamada * PERGUNTAS_POR_ALUNO_MES

itens = [
    ("Resposta", texto),
    ("Tokens de entrada", uso.prompt_token_count),
    ("Tokens de saída", uso.candidates_token_count),
    ("Tokens de thinking", tokens_thinking),
    ("Custo desta chamada (Flash, US$ 1,50/9,00 por milhão)", f"US$ {custo_chamada:.6f}"),
    ("Mesma chamada no Pro (US$ 2,00/12,00 por milhão)",
     f"US$ {uso.prompt_token_count / 1_000_000 * PRECO_PRO_ENTRADA + (uso.candidates_token_count + tokens_thinking) / 1_000_000 * PRECO_PRO_SAIDA:.6f}"),
    ("Mesma chamada no Flash-Lite (US$ 0,25/1,50 por milhão)",
     f"US$ {uso.prompt_token_count / 1_000_000 * PRECO_FLASH_LITE_ENTRADA + (uso.candidates_token_count + tokens_thinking) / 1_000_000 * PRECO_FLASH_LITE_SAIDA:.6f}"),
    ("Servidor GPU próprio na EC2, custo fixo mensal (jul/2026)",
     "; ".join(f"{nome}: US$ {valor}" for nome, valor in SERVIDORES_GPU_MENSAL.items())),
    ("Custo de inferência por aluno/mês no SaaS (hipótese: 100 perguntas)",
     f"US$ {custo_aluno_mes:.2f}"),
]
conclusao = (
    f"mesmo com os preços reais de 2026, a chamada medida custou centavos de dólar. O "
    f"orçamento mensal do servidor GPU mais barato, US$ {CUSTO_SERVIDOR_MENSAL}, pagaria "
    f"cerca de {chamadas_por_mes_do_servidor:,.0f} chamadas iguais — muito acima do volume "
    f"de um projeto de curso — e as instâncias maiores (g6, g5) só aumentam essa distância. "
    f"O Pro custaria ~33% mais por token que o Flash, sem necessidade demonstrada para as "
    f"tarefas curtas deste projeto. Na régua do SaaS do projeto — traduzir as aulas para o "
    f"espanhol e responder perguntas de alunos hispanofalantes — um aluno ativo custa cerca "
    f"de US$ {custo_aluno_mes:.2f} de inferência por mês: qualquer mensalidade modesta cobre "
    f"esse custo com folga, e o custo variável cresce junto com a receita — mais alunos, "
    f"mais chamadas, mais faturamento. Já o servidor próprio começaria todo mês US$ "
    f"{CUSTO_SERVIDOR_MENSAL} no vermelho, exigindo dezenas de assinantes só para empatar. "
    f"É essa estrutura de custo, que só cobra pelo que se usa, que viabiliza o lucro do "
    f"SaaS com inferência remota."
)
mostrar_evidencia(itens, conclusao)

### §3.3 Latência

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — a latência tem duas exigências
bem diferentes. A **tradução** da aula é um trabalho de lote: roda uma vez por aula, ninguém
fica esperando com a tela aberta, e alguns segundos a mais não mudam nada. Já as
**perguntas do aluno** são interativas: a exigência é de segundos, como em qualquer chat —
não de milissegundos.

A chamada remota paga a ida e volta pela internet, algo que a inferência local não tem.
Mesmo assim ela tende a ganhar: o modelo roda em GPUs muito superiores ao hardware do
aluno — processar a transcrição inteira como contexto numa máquina comum levaria minutos,
ou nem carregaria o modelo — e, quando muitos alunos perguntam ao mesmo tempo, a
infraestrutura do Google atende em paralelo, enquanto um servidor próprio único faria fila
e degradaria a latência conforme a carga.

A desvantagem honesta é a variação: rede e carga dos servidores entram na conta e estão
fora do nosso controle — é exatamente o que a célula abaixo mede, cronometrando 5 chamadas
curtas seguidas. Em produção, o SaaS mitigaria a espera percebida com **streaming** (mostrar
a resposta enquanto ela é gerada) e com o **cache de contexto** (a transcrição repetida
entre perguntas não é reprocessada por inteiro — o que também barateia, como visto no
critério de custo).

**Veredito para o nosso caso de uso:** a remota atende com folga as duas cargas do SaaS — segundos de chat para as perguntas do aluno e lote para a tradução — e ainda escala em paralelo quando muitos alunos perguntam ao mesmo tempo, algo que um servidor próprio único não faria.

</div>

In [5]:
import time

pergunta_latencia = "Question: Em que idioma este transcript está escrito?\n\nAnswer in Portuguese, in one word."

tempos = []
for _ in range(5):
    inicio = time.perf_counter()
    gerar(SYSTEM, pergunta_latencia)
    tempos.append(time.perf_counter() - inicio)

tempos_ordenados = sorted(tempos)
mediana = tempos_ordenados[len(tempos_ordenados) // 2]

itens = [
    ("Tempos de cada chamada (s)", ", ".join(f"{t:.2f}" for t in tempos)),
    ("Mínimo", f"{min(tempos):.2f}s"),
    ("Mediana", f"{mediana:.2f}s"),
    ("Máximo", f"{max(tempos):.2f}s"),
]
conclusao = (
    "a variação entre chamadas idênticas mostra que a rede e a carga do servidor entram "
    "na conta, algo fora do nosso controle. Para a régua do SaaS — um aluno esperando a "
    "resposta da sua pergunta — uma mediana de poucos segundos é experiência normal de "
    "chat, e a tradução da aula, feita em lote, nem tem alguém esperando na tela. Em "
    "produção, streaming e cache de contexto reduziriam ainda mais a espera percebida e "
    "a variação."
)
mostrar_evidencia(itens, conclusao)

### §3.4 Disponibilidade

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — disponibilidade significa o
serviço responder na hora em que o aluno estuda: de noite, no fim de semana, na véspera da
prova. O serviço tem duas partes: a nossa aplicação, que é fina, e a inferência por trás
dela, que é a parte pesada — e com a API remota essa parte pesada fica com o Google.

É por isso que a inferência remota ganha neste critério: a API roda em infraestrutura
global, redundante e operada por equipe profissional 24/7 — é muito difícil ela cair. Um
servidor GPU próprio seria um ponto único de falha (uma máquina, uma região), exigindo
manutenção, monitoramento e plantão que um SaaS pequeno não tem como pagar; e uma solução
local só funciona enquanto aquele computador específico estiver ligado.

A desvantagem honesta é a dependência de um terceiro — mas o risco realista não é o
serviço sair do ar, e sim mudanças de catálogo (modelos descontinuados e substituídos), o
que se administra acompanhando os avisos de depreciação; em produção, o SaaS ainda poderia
manter um provedor de reserva.

Um teste único não prova disponibilidade de longo prazo, mas dá para verificar o que
importa agora, com expectativa positiva: a API responde? O modelo usado no projeto está no
catálogo? E uma sequência de chamadas repetidas retorna toda com sucesso?

**Veredito para o nosso caso de uso:** o Google mantém o serviço no ar melhor do que nós conseguiríamos com qualquer servidor próprio — e é isso que garante ao aluno encontrar o SaaS disponível a qualquer hora, sem custo de operação para a gente.

</div>

In [6]:
# Disponibilidade medida agora, em três verificações: a API responde? o modelo do
# projeto aparece no catálogo? e chamadas repetidas retornam todas com sucesso?
try:
    modelos = list(client.models.list())
    nomes_modelos = [m.name or "" for m in modelos]
    api_alcancavel = True
except Exception as erro:
    nomes_modelos = []
    api_alcancavel = False
    print("Erro ao consultar a API:", erro)

modelo_projeto_listado = any(MODELO_GEMINI in nome for nome in nomes_modelos)

# Sequência de chamadas curtas e idênticas (system mínimo, sem o transcript, para custar
# quase nada): conta quantas retornam com sucesso.
SYSTEM_PING = "You are a health-check endpoint."
N_CHAMADAS = 5
sucessos, falhas = 0, []
for _ in range(N_CHAMADAS):
    try:
        gerar(SYSTEM_PING, "Reply with the single word: OK", max_new_tokens=256)
        sucessos += 1
    except Exception as erro:
        falhas.append(str(erro))

itens = [
    ("API respondeu agora", api_alcancavel),
    (f"Modelo do projeto ({MODELO_GEMINI}) aparece no catálogo", modelo_projeto_listado),
    ("Chamadas repetidas bem-sucedidas", f"{sucessos} de {N_CHAMADAS}"),
    ("Falhas observadas", "; ".join(falhas) if falhas else "nenhuma"),
]
resumo_chamadas = (
    "todas as chamadas retornaram com sucesso"
    if sucessos == N_CHAMADAS
    else f"apenas {sucessos} de {N_CHAMADAS} chamadas retornaram com sucesso"
)
conclusao = (
    f"{resumo_chamadas}, com a API alcançável e o modelo do projeto no catálogo. Esse "
    "resultado positivo é o esperado: a API roda na infraestrutura global e redundante do "
    "Google, operada por equipe profissional — é muito difícil ela cair. Para o SaaS, isso "
    "significa que um aluno perguntando na véspera da prova encontra o serviço no ar a "
    "qualquer hora, sem que a gente mantenha servidor nenhum: a responsabilidade pela "
    "parte pesada fica com o Google, e a nossa se reduz a manter a aplicação fina "
    "funcionando. O risco realista não é o serviço sair do ar, e sim mudanças de catálogo "
    "(modelos descontinuados e substituídos), o que se administra acompanhando os avisos "
    "de depreciação."
)
mostrar_evidencia(itens, conclusao)

### §3.5 Controle

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — o controle que importa é
sobre o **comportamento do produto**: a resposta sair do transcript, no idioma certo, no
formato esperado. Esse controle vive no que a API expõe (instruções de sistema,
temperature e seed, limites de tokens) e na camada que é 100% nossa: os prompts, a
validação da saída e as retentativas — tudo já demonstrado nos notebooks anteriores.

O que cedemos, honestamente: a versão exata do modelo, os pesos e o funcionamento
interno. O Google pode atualizar o modelo por trás do mesmo nome, sem aviso. Para o SaaS,
o risco realista desse limite é uma mudança silenciosa no estilo ou na qualidade das
respostas.

A troca compensa porque as tarefas do SaaS — traduzir e responder com contexto — não
precisam de ajuste de pesos nem de fine-tuning; e o risco da mudança silenciosa se
administra com o que já construímos: validadores de formato, extração por marcador e
perguntas de teste conhecidas, que funcionam como teste de regressão do provedor.

Não dá para medir "falta de controle" diretamente, mas dá para mostrar as duas faces do
problema: o quanto a API expõe sobre o modelo, e se duas chamadas idênticas realmente
produzem a mesma resposta.

**Veredito para o nosso caso de uso:** a remota cede um controle fino que o SaaS não precisa (pesos, versão exata) e conserva o controle que importa — prompts, validação da saída e retentativas. É uma troca vantajosa para este produto.

</div>

In [7]:
info_modelo = client.models.get(model=MODELO_GEMINI)

pergunta_determinismo = "Question: Qual é o nome do curso mencionado neste transcript?\n\nAnswer in Portuguese, in one sentence."
resposta_1 = gerar(SYSTEM, pergunta_determinismo)
resposta_2 = gerar(SYSTEM, pergunta_determinismo)

itens = [
    ("Nome do modelo", info_modelo.name),
    ("Limite de tokens de entrada", info_modelo.input_token_limit),
    ("Limite de tokens de saída", info_modelo.output_token_limit),
    ("Resposta 1", resposta_1),
    ("Resposta 2", resposta_2),
    ("Respostas idênticas", resposta_1 == resposta_2),
]
if resposta_1 == resposta_2:
    conclusao = (
        "com temperature=0 e seed=42 a resposta se repete hoje, mas isso vale só para a "
        "versão do modelo servida neste momento; não temos como impedir o Google de "
        "atualizar o modelo por trás do mesmo nome."
    )
else:
    conclusao = (
        "mesmo com temperature=0 e seed=42, a resposta variou — prova direta do limite "
        "de controle já admitido na justificativa acima: seed é melhor esforço, não "
        "garantia."
    )
conclusao += (
    " No SaaS, o controle que importa — prompts, validação da saída e retentativas — "
    "continua 100% nosso; contra atualizações silenciosas do modelo, as perguntas de "
    "teste conhecidas funcionam como teste de regressão do provedor."
)
mostrar_evidencia(itens, conclusao)

### §3.6 Facilidade de integração

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — a nossa parte do produto deve
ser fina: uma aplicação web que chama a inferência. Com a API remota, integrar a
inferência se resume a uma biblioteca pequena (`google-genai`), uma chave em variável de
ambiente e chamadas simples em Python — exatamente o caminho que os notebooks anteriores
já usam. Na prática, o código destes notebooks já é o esqueleto do backend do SaaS.

A alternativa local ou privada carregaria torch, transformers e gigabytes de modelos,
além de gerência de memória, GPU e deploy. Para uma equipe pequena de SaaS, isso é
infraestrutura competindo em tempo com o produto — a tradução das aulas e a qualidade das
respostas ao aluno, que é onde está o valor.

A desvantagem honesta é o acoplamento ao formato e às regras dessa biblioteca e do
provedor. Ela pesa pouco frente ao tempo economizado — e justamente porque a camada de
integração é fina, trocar de provedor, se um dia for preciso, é barato: os prompts e a
validação de saída são portáveis.

Dá para medir a diferença de esforço em números concretos: o tamanho em disco de cada
caminho de instalação, e o cache de modelos que a inferência local já exigiu baixar em
c01 e c02.

**Veredito para o nosso caso de uso:** integrar a inferência custa uma biblioteca e uma chave — a equipe do SaaS gasta o tempo no produto, não em infraestrutura. Ganho claro para a remota.

</div>

In [8]:
import importlib.util


def tamanho_pacote_mb(nome_pacote):
    spec = importlib.util.find_spec(nome_pacote)
    if spec is None or spec.origin is None:
        return 0
    raiz = Path(spec.origin).parent
    total = sum(f.stat().st_size for f in raiz.rglob("*") if f.is_file())
    return total / (1024 * 1024)


tamanho_genai = tamanho_pacote_mb("google.genai")
tamanho_torch = tamanho_pacote_mb("torch")
tamanho_transformers = tamanho_pacote_mb("transformers")

cache_hf = Path.home() / ".cache" / "huggingface"
tamanho_cache_hf = (
    sum(f.stat().st_size for f in cache_hf.rglob("*") if f.is_file()) / (1024 * 1024)
    if cache_hf.exists() else 0
)

itens = [
    ("google-genai instalado", f"{tamanho_genai:.1f} MB"),
    ("torch instalado", f"{tamanho_torch:.1f} MB"),
    ("transformers instalado", f"{tamanho_transformers:.1f} MB"),
    ("Cache de modelos Hugging Face já baixados em c01/c02", f"{tamanho_cache_hf:.1f} MB"),
]
conclusao = (
    "a integração remota custa uma biblioteca pequena e uma chave de API; a integração "
    "local custa muito mais espaço em disco de bibliotecas e modelos baixados, além de "
    "exigir gerenciar memória e dispositivo. Para o SaaS, isso significa que a equipe "
    "gasta o tempo no produto — a tradução das aulas e a qualidade das respostas ao "
    "aluno — em vez de administrar GPU, memória e deploy de modelos; a camada de "
    "integração é tão fina que o código destes notebooks já é, na prática, o esqueleto "
    "do backend."
)
mostrar_evidencia(itens, conclusao)

### §3.7 Limitações de hardware

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — a inferência remota tira o
hardware pesado do nosso caminho nas duas pontas. Na ponta do desenvolvimento, o projeto
inteiro foi feito numa máquina comum, sem GPU dedicada. Na ponta da produção, o backend do
SaaS não hospeda modelo nenhum: só roda a aplicação fina e, com o pipeline de c05, mantém
o índice FAISS em memória — poucos megabytes.

A evidência abaixo mostra o contraste: esta máquina não teria como hospedar um modelo de
linguagem grande (um modelo de 70 bilhões de parâmetros em fp16 pede cerca de 140 GB de
memória), e isso simplesmente não importa, porque a geração roda inteira no Gemini. Por
isso a infraestrutura mínima de produção é uma instância pequena e barata, **sem GPU** —
nada parecido com a g4dn com GPU, que só faria sentido se hospedássemos o próprio modelo.

E é aqui que o critério vira vantagem de escala para o SaaS: mais alunos perguntando não
significa mais hardware nosso — a parte pesada escala do lado do Google. A célula abaixo
verifica o hardware real desta máquina, o que um modelo grande exigiria, e a
infraestrutura mínima do backend em produção.

**Veredito para o nosso caso de uso:** sem GPU em nenhuma ponta, backend mínimo e barato, e mais alunos não exigem mais hardware nosso. Neste critério a remota transforma uma limitação em vantagem de escala.

</div>

In [9]:
import torch

gpu_disponivel = torch.cuda.is_available()
if gpu_disponivel:
    nome_gpu = torch.cuda.get_device_name(0)
    memoria_gpu_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
else:
    nome_gpu = "nenhuma"
    memoria_gpu_gb = 0

# leitura específica de Linux; em Windows ou Mac a RAM total precisaria ser checada de outra forma
with open("/proc/meminfo") as arquivo:
    linha_ram = next(l for l in arquivo if l.startswith("MemTotal"))
ram_total_gb = int(linha_ram.split()[1]) / (1024 * 1024)

parametros_modelo_grande_bilhoes = 70
memoria_necessaria_gb = parametros_modelo_grande_bilhoes * 2  # 2 bytes por parâmetro em fp16

# Instância de produção escolhida: como a geração é toda remota (Gemini), o backend deste
# produto não precisa de GPU nem de muita RAM — só precisa rodar a aplicação e, no futuro,
# manter o índice FAISS de c05 carregado em memória. Uma instância pequena e barata já
# atende; nada parecido com a g4dn.xlarge com GPU discutida em c01/c03, que só faria
# sentido se estivéssemos hospedando o próprio modelo de linguagem.
INSTANCIA_AWS_PRODUCAO = "t4g.medium"  # 2 vCPU, 4 GiB RAM, Graviton (ARM), sem GPU
RAM_INSTANCIA_GB = 4

# Tamanho estimado do índice FAISS do corpus atual (c03): ~2173 vetores de 768 dimensões,
# float32 (4 bytes cada) — dá para estimar sem precisar carregar o índice aqui.
vetores_indice_estimado = 2173
dimensao_indice = 768
memoria_indice_mb = (vetores_indice_estimado * dimensao_indice * 4) / (1024 ** 2)

itens = [
    ("GPU disponível nesta máquina", f"{gpu_disponivel} - {nome_gpu}"),
    ("Memória de GPU disponível", f"{memoria_gpu_gb:.1f} GB"),
    ("RAM total desta máquina", f"{ram_total_gb:.1f} GB"),
    (
        f"Memória necessária para um modelo de {parametros_modelo_grande_bilhoes}B em fp16",
        f"{memoria_necessaria_gb} GB",
    ),
    ("Instância AWS escolhida para produção", f"{INSTANCIA_AWS_PRODUCAO} ({RAM_INSTANCIA_GB} GiB RAM, sem GPU, cerca de US$0,03 a US$0,04/hora)"),
    ("Memória estimada do índice FAISS atual (c03)", f"cerca de {memoria_indice_mb:.1f} MB"),
]
conclusao = (
    "esta máquina não teria como hospedar um modelo de linguagem desse porte, mas isso não "
    f"importa em produção: como a geração roda inteiramente no Gemini, uma instância pequena "
    f"e barata como a {INSTANCIA_AWS_PRODUCAO} já basta, sem GPU nenhuma, bem diferente da "
    "g4dn.xlarge discutida em c01/c03, que só faria sentido para hospedar um modelo próprio. "
    "O único uso relevante de memória local seria o índice FAISS que c05 vai carregar: no "
    "tamanho atual do corpus, poucos megabytes, bem abaixo dos 4 GiB de RAM dessa instância — "
    "só passaria a exigir uma instância maior se o número de aulas ou clientes crescesse "
    "várias ordens de grandeza. Para o SaaS, isso vale nas duas pontas: nem quem "
    "desenvolve nem quem paga o backend precisa de GPU — e mais alunos perguntando não "
    "significa mais hardware nosso, porque a parte pesada escala do lado do Google."
)
mostrar_evidencia(itens, conclusao)

### §3.8 Dependência de internet

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — este critério muda de peso:
um SaaS é, por definição, um produto de internet. O aluno já acessa o serviço pelo
navegador; sem conexão, ele não chega nem à nossa página, com ou sem inferência remota.
Ou seja, para este produto a dependência de internet não é um custo extra da inferência
remota — ela já era uma premissa do negócio.

Do lado do backend, em produção na nuvem, a conectividade é responsabilidade contratual
do provedor — a conclusão da célula abaixo cita o SLA da AWS para a instância escolhida
em §3.7. A falha de conexão só é um problema visível no ambiente de desenvolvimento, a
máquina de um aluno, que pode perder o wifi. A ressalva honesta: uma solução local com
Ollama continuaria funcionando offline — mas isso só ajudaria num produto de desktop, não
num SaaS, que sem internet não tem como ser acessado de qualquer forma.

Dá para verificar a conexão de duas formas: confirmando que uma chamada real à API
funciona com a conexão normal desta máquina, e provocando de verdade uma falha de conexão,
sem mexer na rede do sistema, com um cliente apontando para uma porta local fechada.

**Veredito para o nosso caso de uso:** para um SaaS — um produto que já vive na internet —
a dependência de conexão não adiciona risco novo: o aluno já precisa de rede para abrir o
serviço, e no backend a conectividade é garantida por SLA do provedor de nuvem. Neste
critério, a inferência remota não perde nada que o produto já não assumisse.

</div>

In [10]:
try:
    resposta_teste_online = client.models.generate_content(
        model=MODELO_GEMINI, contents="teste de conectividade, responda apenas OK"
    )
    chamada_online_funcionou = bool(resposta_teste_online.text)
except Exception as erro:
    chamada_online_funcionou = False

cliente_offline = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(base_url="http://127.0.0.1:9", timeout=3000),
)

try:
    cliente_offline.models.generate_content(model=MODELO_GEMINI, contents="teste")
    chamada_offline_falhou = False
    tipo_erro_offline = "nenhum (chamada funcionou inesperadamente)"
except Exception as erro:
    chamada_offline_falhou = True
    tipo_erro_offline = type(erro).__name__

itens = [
    ("Chamada real à API, com conexão normal desta máquina", "funcionou" if chamada_online_funcionou else "falhou"),
    ("Chamada sem rota de rede real falhou como esperado", chamada_offline_falhou),
    ("Tipo do erro capturado na chamada offline", tipo_erro_offline),
]
conclusao = (
    "com conexão normal a chamada funciona sem problema, e sem conexão ela falha de "
    "verdade, como mostrado acima — o contraste confirma a dependência de internet já "
    "descrita na justificativa. Essa falha só aparece no ambiente de desenvolvimento, a "
    "máquina de um aluno; <b>num cenário real de produção na nuvem, como uma instância "
    "gerenciada na AWS, a conectividade é estável e cuidada pelo provedor, então esse "
    "tipo de problema não deveria ocorrer</b>. Isso não é só uma suposição: o "
    "<a href=\"https://aws.amazon.com/compute/sla/\">Compute Service Level Agreement da "
    "AWS</a> garante, para uma única instância EC2 como a t4g.medium escolhida em §3.7, "
    "pelo menos *\"an Instance-Level Uptime Percentage of at least 99.5%\"* por mês, e "
    "define a própria indisponibilidade dessa instância como perda de conectividade "
    "externa (*\"your Single EC2 Instance has no external connectivity\"*) — ou seja, a "
    "conectividade da instância é exatamente o que a AWS se compromete contratualmente a "
    "manter, com créditos de serviço se isso falhar. Para o SaaS, a dependência de "
    "internet já era premissa do produto: o aluno só chega até ele pela rede."
)
mostrar_evidencia(itens, conclusao)

### §3.9 Risco de exposição de dados

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Na régua do **SaaS** deste projeto — traduzir as transcrições das aulas do português para
o espanhol e deixar o aluno hispanofalante fazer perguntas — dois tipos de dado viajam
para a API: as **transcrições das aulas**, a matéria-prima do produto, um conteúdo já
compartilhado entre alunos e professor; e as **perguntas do aluno**, que são sobre o
conteúdo da aula, não dados pessoais. E há um terceiro ativo que não viaja, mas pode
vazar: a **chave de API**, que no SaaS é um segredo do backend — nunca do aluno.

A frente da chave se controla com prática simples, e é exatamente o que a célula abaixo
verifica: o `.env` fora do controle de versão, a chave carregada sem nunca ser impressa.
Em produção, o SaaS usaria um gerenciador de segredos do provedor de nuvem, o mesmo
princípio com mais proteção.

A frente do dado enviado não dá para verificar com código, mas os
[termos de uso da API do Gemini](https://ai.google.dev/gemini-api/terms)
deixam isso escrito: este projeto usa uma chave paga ("Paid Services"), e para esse caso
os termos dizem que *"Google doesn't use your prompts...or responses to improve our
products"* e que *"Google logs prompts and responses for a limited period of time,
solely for detecting and preventing violations"* — ou seja, ao contrário do tier
gratuito, o conteúdo enviado não é usado para melhorar o produto nem passa por revisão
humana de rotina. Para transcrições de aula e perguntas acadêmicas, sem dado pessoal
sensível, essa condição contratual é suficiente — e a regra do SaaS é direta: nada
confidencial vai para a API. A ressalva honesta continua: os mesmos termos dizem que
*"data may be stored transiently or cached in any country in which Google or its agents
maintain facilities"*, então não é privacidade total — o dado passa e pode ficar
armazenado temporariamente em servidores do Google, fora do nosso controle.

**Veredito para o nosso caso de uso:** para os dados que este SaaS realmente movimenta —
transcrições já compartilhadas e perguntas acadêmicas — o tier pago oferece condições
contratuais suficientes, e o risco da chave se controla com as práticas simples
verificadas abaixo. A remota continua sendo a escolha conveniente, com a mesma ressalva
da privacidade (§3.1): material confidencial forçaria a revisitar a escolha.

</div>

In [11]:
import subprocess

resultado_ignore = subprocess.run(
    ["git", "check-ignore", ".env"], capture_output=True, text=True, cwd=Path.cwd()
)
env_ignorado = resultado_ignore.returncode == 0

resultado_ls = subprocess.run(
    ["git", "ls-files", ".env"], capture_output=True, text=True, cwd=Path.cwd()
)
env_rastreado = bool(resultado_ls.stdout.strip())

chave_carregada = bool(GEMINI_API_KEY)

itens = [
    (".env está ignorado pelo git", env_ignorado),
    (".env está rastreado no repositório", env_rastreado),
    ("Chave carregada com sucesso", chave_carregada),
    ("Comprimento da chave", len(GEMINI_API_KEY) if chave_carregada else 0),
]
conclusao = (
    "o arquivo com a chave está protegido pelo git e a chave em si nunca aparece "
    "impressa nesta saída, só o fato de que foi carregada. O que o Google faz com os "
    "dados enviados não dá para verificar com código, isso fica sob os termos de uso do "
    "serviço, não sob nosso controle. No SaaS, a chave viveria só no backend (num "
    "gerenciador de segredos do provedor), e o que o aluno envia — perguntas sobre a "
    "aula — passa sob os termos pagos citados acima; a regra continua a mesma: nada "
    "confidencial vai para a API."
)
mostrar_evidencia(itens, conclusao)

---

## §4 Dependências, variáveis de ambiente e cuidados de segurança

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**A decisão deste notebook: inferência remota, com a API do Gemini Flash.** Os nove
critérios da seção anterior apontam para o mesmo lugar. Para o SaaS deste projeto —
traduzir as transcrições das aulas do português para o espanhol e responder as
perguntas do aluno hispanofalante — a inferência remota vence com folga em custo,
latência, disponibilidade, facilidade de integração e hardware; faz uma troca
vantajosa em controle; e perde pouco, de forma aceitável para este tipo de dado, em
privacidade e exposição de dados. Nenhum critério exige um modelo local ou privado —
e vários deles tornariam o produto inviável sem a API. O que essa escolha exige na
prática é justamente o que esta seção documenta: uma pegada mínima de dependências e
um conjunto pequeno de cuidados de segurança.

Para usar a API do Gemini neste notebook, precisamos de duas bibliotecas que o projeto já usa desde os notebooks anteriores e que estão no requirements.txt:

- **google-genai** — a biblioteca oficial do Google para conversar com a API do Gemini. É por ela que criamos o cliente e enviamos os pedidos de geração de texto.
- **python-dotenv** — uma biblioteca pequena que lê variáveis de ambiente a partir de um arquivo de texto, para não precisarmos escrever segredos dentro do código.

A única variável de ambiente necessária é a **GEMINI_API_KEY**, que é a chave de acesso obtida no Google AI Studio. Ela fica guardada num arquivo chamado `.env` na raiz do projeto, com uma linha simples no formato `GEMINI_API_KEY=valor_da_chave`. No começo do notebook, a função `load_dotenv` carrega esse arquivo e a chave passa a estar disponível no ambiente, sem nunca aparecer no código. O repositório não inclui
nenhuma chave: quem for rodar o projeto — neste caso, o professor, ao avaliar o
trabalho — usa a própria chave, obtida no Google AI Studio, criando o seu `.env`
local com essa única linha. Assim, cada execução usa a conta e a cota de quem executa, e nenhum segredo viaja junto com o código.

Alguns cuidados práticos evitam que a chave ou dados do projeto vazem:

- **Nunca commitar o `.env` no controle de versão.** O projeto acabou de criar um `.gitignore` exatamente para isso: o arquivo `.env` está listado ali e o git passa a ignorá-lo, então ele não entra em nenhum commit por descuido.
- **Nunca escrever a chave direto no código.** A chave só existe no `.env` e é lida pelo ambiente. Também não devemos imprimi-la na saída de uma célula, nem de propósito nem por acidente — a saída das células fica salva dentro do arquivo do notebook, e o notebook vai para o repositório.
- **Trocar a chave imediatamente se ela vazar.** Se por acidente a chave for enviada a um repositório público, apagar o commit não resolve: o histórico pode ter sido copiado. O caminho certo é entrar no Google AI Studio, revogar a chave antiga e gerar uma nova na hora.
- **Não mandar conteúdo confidencial para a API.** Tudo que enviamos ao Gemini passa pelos servidores do Google. Neste projeto só enviamos transcrições de aula, que não são sigilosas; qualquer dado confidencial ou sem autorização de uso fica fora das chamadas à API.

A seção "Evidências por critério" acima já verifica com código, e não só em texto, que o `.env` está ignorado pelo git e não está rastreado no repositório, e que a chave nunca aparece impressa em nenhuma saída.

</div>

---

## §5 Conclusão

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 Por que a API do Gemini Flash é a escolha certa para este projeto, critério a critério**

O caminho deste notebook foi comparar três formas de rodar um modelo de linguagem — **local**,
**remota** e **privada** —, rodar a inferência remota na prática e reunir evidências critério
por critério, sempre sob a régua de um SaaS: privacidade, custo, latência, disponibilidade,
controle, integração, hardware, dependência de internet e risco de exposição de dados. Esta
conclusão junta o que essa comparação mostrou, em três perguntas: o que pesou a favor da API,
onde ela cobra um preço, e o que isso decide para as próximas etapas.

**O que pesou a favor da inferência remota.** Três critérios decidem quase sozinhos, no volume
de um projeto de curso: **custo**, paga-se só pelos tokens usados, contra o custo fixo de um
servidor sempre ligado; **hardware**, a inferência remota tira a GPU do caminho nas duas
pontas, não precisamos de placa de vídeo nem para traduzir nem para responder; e **integração**,
a nossa parte fica fina, uma aplicação que chama a API, sem baixar nem servir modelo. A
**latência** completa o quadro: o Flash é o tier rápido da família, adequado à pergunta
interativa do aluno, enquanto a tradução roda em lote, fora do caminho crítico.

<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
✅ <strong>Exemplo:</strong> no critério de <strong>custo</strong>, uma aula inteira
traduzida saiu por <em>centavos</em>, enquanto um servidor próprio com GPU (AWS
<code>g4dn.xlarge</code>) custaria <em>US$384 a US$400 por mês, fixo, mesmo parado</em> —
pagar por token compensa muito mais neste volume.
</div>

**Onde a API cobra um preço.** O trade-off honesto está na **privacidade** e na **dependência
de terceiros**. Com inferência remota, o texto sai da máquina e viaja até os servidores do
Google; e não temos como impedir o provedor de atualizar o modelo por trás do mesmo nome, nem
garantir 100% de reprodutibilidade — o `seed=42` é melhor esforço. Um SaaS é, por definição,
um produto de internet, então a dependência de conexão deixa de ser um problema; mas a
privacidade só é aceitável **por causa do tipo de dado** deste projeto.

<div style="background:#fff8e6;border-left:4px solid #d9a404;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
⚠️ <strong>Exemplo:</strong> no critério de <strong>privacidade</strong>, os textos saem do
computador e viajam até o Google — aceitável aqui só porque são <em>transcrições de aula</em>,
já compartilhadas e sem dados pessoais sensíveis; se o projeto lidasse com informação
confidencial, essa escolha teria que ser revista.
</div>

**O controle que fica conosco.** O que mais importa não escapa para o provedor: os prompts, a
validação da saída e as retentativas continuam 100% nossos, e o limite de contexto medido ao
vivo (cerca de 1 milhão de tokens de entrada) mostra que o transcript inteiro cabe com folga.
Contra atualizações silenciosas do modelo, perguntas de teste conhecidas funcionam como teste
de regressão do provedor.

**Implicações para as próximas etapas.** Esta decisão — inferência remota com o Gemini Flash —
não fica só neste notebook: é a mesma que C03, nos embeddings e na recuperação, e C05, no RAG,
herdam e colocam para trabalhar, sobre a mesma API e com os mesmos parâmetros de determinismo.
A régua de escolha, critério por critério e sob a ótica de um produto real, é o que sustenta
essa continuidade — cerrando o notebook onde ele começou: a inferência remota é a peça certa
para este caso de uso.

</div>